In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, r2_score
from sklearn.linear_model import LinearRegression
import os

## Load Datasets

In [3]:
base_path = os.pardir + "" + "\data\loan_default_datasets"

demographics = pd.read_csv(os.path.join(base_path, "traindemographics.csv"))
prevloans = pd.read_csv(os.path.join(base_path, "trainprevloans.csv"))
performance = pd.read_csv(os.path.join(base_path, "trainperf.csv"))


## Data Architecture & Merging

In [4]:
demographics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4346 entries, 0 to 4345
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  4346 non-null   object 
 1   birthdate                   4346 non-null   object 
 2   bank_account_type           4346 non-null   object 
 3   longitude_gps               4346 non-null   float64
 4   latitude_gps                4346 non-null   float64
 5   bank_name_clients           4346 non-null   object 
 6   bank_branch_clients         51 non-null     object 
 7   employment_status_clients   3698 non-null   object 
 8   level_of_education_clients  587 non-null    object 
dtypes: float64(2), object(7)
memory usage: 305.7+ KB


In [5]:
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4368 entries, 0 to 4367
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customerid     4368 non-null   object 
 1   systemloanid   4368 non-null   int64  
 2   loannumber     4368 non-null   int64  
 3   approveddate   4368 non-null   object 
 4   creationdate   4368 non-null   object 
 5   loanamount     4368 non-null   float64
 6   totaldue       4368 non-null   float64
 7   termdays       4368 non-null   int64  
 8   referredby     587 non-null    object 
 9   good_bad_flag  4368 non-null   object 
dtypes: float64(2), int64(3), object(5)
memory usage: 341.4+ KB


In [6]:
prevloans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18183 entries, 0 to 18182
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       18183 non-null  object 
 1   systemloanid     18183 non-null  int64  
 2   loannumber       18183 non-null  int64  
 3   approveddate     18183 non-null  object 
 4   creationdate     18183 non-null  object 
 5   loanamount       18183 non-null  float64
 6   totaldue         18183 non-null  float64
 7   termdays         18183 non-null  int64  
 8   closeddate       18183 non-null  object 
 9   referredby       1026 non-null   object 
 10  firstduedate     18183 non-null  object 
 11  firstrepaiddate  18183 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 1.7+ MB


In [7]:
prevloans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18183 entries, 0 to 18182
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       18183 non-null  object 
 1   systemloanid     18183 non-null  int64  
 2   loannumber       18183 non-null  int64  
 3   approveddate     18183 non-null  object 
 4   creationdate     18183 non-null  object 
 5   loanamount       18183 non-null  float64
 6   totaldue         18183 non-null  float64
 7   termdays         18183 non-null  int64  
 8   closeddate       18183 non-null  object 
 9   referredby       1026 non-null   object 
 10  firstduedate     18183 non-null  object 
 11  firstrepaiddate  18183 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 1.7+ MB


In [8]:
prevloans_agg = prevloans.copy()

In [29]:
datetime_Columns = ['firstduedate', 'firstrepaiddate', 'creationdate', 'approveddate', 'closeddate']

def convert_to_date(cols: list, df):
    for key in cols:
        df[key] = pd.to_datetime(df[key])

convert_to_date(datetime_Columns, prevloans_agg)


In [30]:
# result = prevloans_agg.groupby('customerid').apply(lambda x: x['firstrepaiddate'] - x['firstduedate'])
print(prevloans_agg.dtypes)

customerid                     object
systemloanid                    int64
loannumber                      int64
approveddate           datetime64[ns]
creationdate           datetime64[ns]
loanamount                    float64
totaldue                      float64
termdays                        int64
closeddate             datetime64[ns]
referredby                     object
firstduedate           datetime64[ns]
firstrepaiddate        datetime64[ns]
days_late                       int64
expected_close_date    datetime64[ns]
dtype: object


In [22]:
prevloans_agg['days_late'] = (prevloans_agg['firstrepaiddate'] - prevloans_agg['firstduedate']).dt.days
prevloans_agg['days_late']

0       -13
1        -4
2        22
3         0
4        11
         ..
18178    -3
18179    -6
18180    -3
18181    19
18182   -15
Name: days_late, Length: 18183, dtype: int64

In [27]:
prevloans_agg['expected_close_date'] = pd.to_datetime((prevloans_agg['approveddate'] + pd.to_timedelta(prevloans_agg['termdays'], 'D')).dt.date)
prevloans_agg['expected_close_date']

0       2016-09-14
1       2017-05-28
2       2017-04-04
3       2017-04-24
4       2017-07-02
           ...    
18178   2016-05-16
18179   2016-12-18
18180   2016-07-12
18181   2016-09-26
18182   2016-10-14
Name: expected_close_date, Length: 18183, dtype: datetime64[ns]

In [32]:
print(prevloans_agg['expected_close_date'].dtype)
print(prevloans_agg['closeddate'].dtype)

datetime64[ns]
datetime64[ns]


In [ ]:
datetime_Columns = ['firstduedate', 'firstrepaiddate', 'creationdate', 'approveddate', 'closeddate']

def convert_to_date(cols: list, df):
    for key in datetime_Columns:
        df[key] = pd.to_datetime(df[key], errors='coerce')

convert_to_date(datetime_Columns, prevloans)


In [33]:
prevloans_agg['total_settlement_delay'] = prevloans_agg['closeddate'] - prevloans_agg['expected_close_date']

In [35]:
prevloans_agg.columns.to_list()

['customerid',
 'systemloanid',
 'loannumber',
 'approveddate',
 'creationdate',
 'loanamount',
 'totaldue',
 'termdays',
 'closeddate',
 'referredby',
 'firstduedate',
 'firstrepaiddate',
 'days_late',
 'expected_close_date',
 'total_settlement_delay']

In [36]:
prevloans_agg = prevloans_agg.groupby('customerid')

In [ ]:
prevloans_agg = prevloans_agg.agg(
    num_prev_loans=("systemloanid", 'count'),
    total_loan_amount=("loanamount", 'sum'),
    avg_loan_size=('loanamount', 'median'),
    avg_first_installment_delay=(("days_late", 'mean'))
    )

In [38]:
prevloans_agg.head()

,num_prev_loans,total_loan_amount,avg_loan_size,avg_first_installment_delay
customerid,,,,
8a1088a0484472eb01484669e3ce4e0b,1,10000.0,10000.0,6.000000
8a1a1e7e4f707f8b014f797718316cad,4,70000.0,15000.0,-0.250000
8a1a32fc49b632520149c3b8fdf85139,7,90000.0,10000.0,-0.428571
8a1eb5ba49a682300149c3c068b806c7,8,130000.0,15000.0,-3.125000
8a1edbf14734127f0147356fdb1b1eb2,2,20000.0,10000.0,-4.000000


In [40]:
prevloans.columns.to_list()

['customerid',
 'systemloanid',
 'loannumber',
 'approveddate',
 'creationdate',
 'loanamount',
 'totaldue',
 'termdays',
 'closeddate',
 'referredby',
 'firstduedate',
 'firstrepaiddate']

In [ ]:
# cols = ["date", 'birth', 'creation', "approved"]
# df_cols = prevloans.columns.to_list()
# # for col in cols:
# #     i = 0
# #     if col in df_cols[i].lower():
# #         print(True)
# #         i += 1

# for col in df_cols:



In [46]:
prevloans.select_dtypes(include=[np.number]).columns


Index(['systemloanid', 'loannumber', 'loanamount', 'totaldue', 'termdays'], dtype='object')

In [50]:
prevloans.select_dtypes(include=['category', 'object']).columns

Index(['customerid', 'closeddate', 'referredby'], dtype='object')